**<h1>Electric Line Extension - Analysis 4**
#### Descriptive Data
<i>Any question regarding the notebook, please contact Robert Ford<br>
    Last Updated: 04/08/2026 | Start Development: 04/07/2026</i>
* Utility / IOU Data from PG&E, SDG&E, and SCE for Q1(Revised) 2025 - Q3 2025
* Goals: 1. Clean excel files for melting and future uploading to Tableau. See 'Clean 4' below for full details and results
*           Update tables as needed.


In [2]:
# Import Libraries and Folders
import pandas as pd
import os
from pathlib import Path

# ============================================
# SET YOUR INPUT / OUTPUT FOLDERS
# ============================================

input_folder = r"C:/Users/Rford/OneDrive - California Energy Commission/Documents/Analysis - Scripts and Code/Electric-Line-Extension-Data/data/raw"
output_folder = r"C:/Users/Rford/OneDrive - California Energy Commission/Documents/Analysis - Scripts and Code/Electric-Line-Extension-Data/data/processed"

os.makedirs(output_folder, exist_ok=True)


In [3]:
# ============================================
# FILE MAP
# ============================================

file_info = {
    #"PG&E_Q1_2025 Report_Revised.xlsx": {"quarter": "Q1", "iou": "PG&E", "year": 2025},
    #"PG&E_Q3_2025 Report.xlsx": {"quarter": "Q3", "iou": "PG&E", "year": 2025},
    #"PGE Q2_2025 Report.xlsx": {"quarter": "Q2", "iou": "PG&E", "year": 2025},
    "PG&E_Q4_2025.xlsx": {"quarter": "Q4", "iou": "PG&E", "year": 2025},

    #"SCE_Q1_2025 REport Supplemental.xlsx": {"quarter": "Q1", "iou": "SCE", "year": 2025},
    #"SCE Q2_2025_Report.xlsx": {"quarter": "Q2", "iou": "SCE", "year": 2025},
    #"SCE_Q3_2025 Report.xlsx": {"quarter": "Q3", "iou": "SCE", "year": 2025},
    "SCE_Q4_2025 Report and Annual Summary.xlsx": {"quarter": "Q4", "iou": "SCE", "year": 2025},

    #"SDG&E_Q1 2025 Report Supplemental.xlsx": {"quarter": "Q1", "iou": "SDG&E", "year": 2025},
    #"SDGE Q2_2025_Report.xlsx": {"quarter": "Q2", "iou": "SDG&E", "year": 2025},
    #"SDG&E_Q3 2025 Report.xlsx": {"quarter": "Q3", "iou": "SDG&E", "year": 2025},
    "SDGE_Q4_2025 Report and Annual Report.xlsx": {"quarter": "Q4", "iou": "SDG&E", "year": 2025},
}


In [4]:
# ============================================
# COLUMN STANDARDIZATION FUNCTION
# ============================================

def clean_column_names(cols):
    cleaned = []
    for c in cols:
        c = str(c).strip()
        c = c.replace("\n", " ")
        c = " ".join(c.split())
        c = c.replace("All All Electric", "All Electric")
        cleaned.append(c)
    return cleaned

In [5]:
# ============================================
# FIND HEADER ROW AUTOMATICALLY
# ============================================

def detect_header_row(filepath):
    preview = pd.read_excel(filepath, header=None, nrows=20)

    for i in range(len(preview)):
        row_values = preview.iloc[i].astype(str).tolist()
        joined = " ".join(row_values)

        if "Customer Class" in joined and "Month" in joined:
            return i

    return 0

In [6]:
# ============================================
# LOAD + CLEAN EACH FILE
# ============================================

quarter_data = {
    #"Q1": [],
    #"Q2": [],
    #"Q3": [],
    "Q4": []
}

for filename, meta in file_info.items():
    filepath = os.path.join(input_folder, filename)

    print(f"Processing: {filename}")

    header_row = detect_header_row(filepath)

    df = pd.read_excel(filepath, header=header_row)

    # Clean columns
    df.columns = clean_column_names(df.columns)
    
    df.columns = [
        col.replace("All Electric Line or Service", "Electric Line or Service")
        for col in df.columns
    ]

    # Drop completely empty rows
    df = df.dropna(how='all')

    # Add missing Year column if needed
    if "Year" not in df.columns:
        df["Year"] = meta["year"]

    # Add IOU column
    df["IOU"] = meta["iou"]

    # Reorder columns so Year + IOU come first
    cols = df.columns.tolist()
    cols.remove("Year")
    cols.remove("IOU")
    df = df[["Year", "IOU"] + cols]

    # Append to quarter group
    quarter_data[meta["quarter"]].append(df)

Processing: PG&E_Q4_2025.xlsx
Processing: SCE_Q4_2025 Report and Annual Summary.xlsx
Processing: SDGE_Q4_2025 Report and Annual Report.xlsx


In [7]:
# ============================================
# COMBINE AND EXPORT EACH QUARTER
# ============================================

for quarter, dfs in quarter_data.items():
    combined_df = pd.concat(dfs, ignore_index=True)

    output_path = os.path.join(output_folder, f"Condensed_{quarter}_2025.xlsx")
    combined_df.to_excel(output_path, index=False)

    print(f"Saved: {output_path}")

print("All quarterly condensed files created successfully.")

PermissionError: [Errno 13] Permission denied: 'C:/Users/Rford/OneDrive - California Energy Commission/Documents/Analysis - Scripts and Code/Electric-Line-Extension-Data/data/processed\\Condensed_Q4_2025.xlsx'

In [9]:
# Paths to your cleaned quarterly files
q1_file = os.path.join("C:/Users/Rford/OneDrive - California Energy Commission/Documents/Analysis - Scripts and Code/Electric-Line-Extension-Data/data/processed", "Condensed_Q1_2025.xlsx")
q2_file = os.path.join("C:/Users/Rford/OneDrive - California Energy Commission/Documents/Analysis - Scripts and Code/Electric-Line-Extension-Data/data/processed", "Condensed_Q2_2025.xlsx")
q3_file = os.path.join("C:/Users/Rford/OneDrive - California Energy Commission/Documents/Analysis - Scripts and Code/Electric-Line-Extension-Data/data/processed", "Condensed_Q3_2025.xlsx")
q4_file = os.path.join("C:/Users/Rford/OneDrive - California Energy Commission/Documents/Analysis - Scripts and Code/Electric-Line-Extension-Data/data/processed", "Condensed_Q4_2025.xlsx")

# Load quarterly datasets
df_q1 = pd.read_excel(q1_file)
df_q2 = pd.read_excel(q2_file)
df_q3 = pd.read_excel(q3_file)
df_q4 = pd.read_excel(q4_file)

# Combine all quarters into master dataframe
master_df = pd.concat([df_q1, df_q2, df_q3, df_q4], ignore_index=True)

# Save master file
master_output_path = os.path.join("C:/Users/Rford/OneDrive - California Energy Commission/Documents/Analysis - Scripts and Code/Electric-Line-Extension-Data/data/processed", "Master_Q1-Q4_2025.xlsx")
master_df.to_excel(master_output_path, index=False)

print(f"Master file created successfully: {master_output_path}")

Master file created successfully: C:/Users/Rford/OneDrive - California Energy Commission/Documents/Analysis - Scripts and Code/Electric-Line-Extension-Data/data/processed\Master_Q1-Q4_2025.xlsx
